# StoryZop: Instagram Story Analyzer
This notebook runs the complete StoryZop pipeline on Google Colab, leveraging Playwright for browser automation and Qwen3-VL models for vision analysis.

## Section 1: Install Dependencies
Install the required libraries for browser automation, AI models, and OCR.

In [ ]:
!pip install playwright Pillow pydantic pydantic-settings python-dotenv nest-asyncio
!playwright install chromium
!pip install torch transformers accelerate qwen-vl-utils bitsandbytes easyocr
!git clone https://github.com/10Unknownboy/StoryZop.git
%cd StoryZop

import nest_asyncio
nest_asyncio.apply()

## Section 2: GPU/Environment Check
Check the current GPU allocation to ensure the models will fit in VRAM.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.vision.gpu import GPUManager
GPUManager.print_gpu_status()

## Section 3: Configuration
Set the core configuration values for the pipeline. We use Colab userdata (Secrets) to store the Instagram sessionid securely.

In [ ]:
from src.config import get_config
from google.colab import userdata

try:
    # Create a secret in Colab on the left menu (key icon) called INSTAGRAM_SESSIONID
    session_id = userdata.get('INSTAGRAM_SESSIONID')
except:
    print("\u26a0\ufe0f Colab secret 'INSTAGRAM_SESSIONID' not found. Please set it in the Secrets tab.")
    session_id = None

config = get_config(
    instagram_sessionid=session_id,
    use_4bit_quantization=True,
    headless=True
)

print("Configured Data Directory:", config.data_dir)
print("Configured Models:", config.initial_model, "|", config.primary_model)

## Section 4: Load Models
Initialize the Qwen3-VL models. Loading 32B might require high-RAM and A100.

In [ ]:
from src.vision.qwen4b import Qwen4BScreener
from src.vision.qwen8b import Qwen8BAnalyzer
from src.vision.qwen32b import Qwen32BExpert
from src.vision.ocr import OCREngine

print("Initializing models (lazy loading)...")
screener = Qwen4BScreener(config)
analyzer = Qwen8BAnalyzer(config)
expert = Qwen32BExpert(config)
ocr_engine = OCREngine()

# Optional: load them now into VRAM, or let them load lazily when used
# screener.load_model()

## Section 5: Initialize Database
Initialize the local SQLite database.

In [ ]:
from src.database.database import Database

db = Database(config.db_path)
db.initialize()

state = db.get_processing_state()
print("Database initialized. State:", state)

## Section 6: Initialize Browser & Authenticate
Launch the Playwright browser and load the sessionid cookie to authenticate.

In [ ]:
from src.browser.session import BrowserSession
from src.browser.instagram import InstagramNavigator
from src.browser.stories import StoryNavigator
from src.capture.frame_manager import FrameManager
from src.capture.sampler import StorySampler

session = BrowserSession(config)
await session.launch()

if config.instagram_sessionid:
    await session.load_sessionid(config.instagram_sessionid)
else:
    print("\u26a0\ufe0f Running without session ID. Instagram will block story access.")

navigator = InstagramNavigator(session.page, config)
story_nav = StoryNavigator(session.page, config)
frame_manager = FrameManager(config)
sampler = StorySampler(config, frame_manager)

await navigator.navigate_to_instagram()
is_auth = await navigator.verify_authentication()
print(f"Authenticated: {is_auth}")

await navigator.dismiss_dialogs()

## Section 7: Run End-to-End Pipeline
Scan stories, run the initial analysis, process revisit requests, run final analysis, and export.

In [ ]:
from src.pipeline import StoryPipeline

pipeline = StoryPipeline(config, db)
pipeline.set_browser(session)
pipeline.set_models(screener, analyzer, expert)
pipeline.set_ocr(ocr_engine)
pipeline.set_components(navigator, sampler)

print("Starting pipeline run...")
stats = await pipeline.run()
print("Pipeline execution completed! Stats:", stats)

## Section 8: Export Results
Generate text reports, JSON, and CSV data.

In [ ]:
from src.analysis.report import ReportGenerator

report_gen = ReportGenerator(db)

print("--- Text Report ---")
print(report_gen.generate_text_report())

json_path = config.data_dir / "export.json"
csv_path = config.data_dir / "export.csv"

report_gen.export_json(json_path)
report_gen.export_csv(csv_path)

print(f"\nExported JSON to {json_path}")
print(f"Exported CSV to {csv_path}")

# Cleanup browser
await session.close()